In [ ]:
import os
from pathlib import Path

root_dir = Path.home() / "Github" / "spatial_ml"
data_dir = root_dir / "data" / "training" / "zoning_segmentation"

assert data_dir.exists(), f"Data directory {data_dir} does not exist"
print(f"Data directory contents: {os.listdir(data_dir)}")

In [ ]:
import json
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

from alignment import align_geojson_to_image, count_visible_features, geo_to_pixel, load_sample

In [ ]:
def show_image_and_geojson(sample_id: str):
    """Show map image (left) and aligned GeoJSON (right) side by side."""
    ann, gdf = load_sample(data_dir, sample_id)
    img = Image.open(data_dir / "images" / f"{sample_id}.png")

    zone_col = ann.get("zone_column", "Zoning")
    zone_styles = ann.get("zone_styles", {})
    rotation = ann.get("rotation", 0.0)
    img_w, img_h = ann.get("image_width", img.size[0]), ann.get("image_height", img.size[1])

    # Align GeoJSON to rendered image
    plot_gdf, viewport = align_geojson_to_image(ann, gdf)
    xmin, ymin, xmax, ymax = viewport

    # Count features in viewport
    stats = count_visible_features(plot_gdf, viewport, zone_col)

    # Build color map
    color_map = {name: style.get("color", "#cccccc") for name, style in zone_styles.items()}

    # Plot
    img_aspect = img_h / img_w
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10 * img_aspect))

    ax1.imshow(img)
    ax1.set_title(f"Map Image: {sample_id}", fontsize=14)
    ax1.set_xticks([]); ax1.set_yticks([])
    for spine in ax1.spines.values():
        spine.set_visible(True); spine.set_linewidth(1.5)

    if zone_col in plot_gdf.columns:
        for zone_name, group in plot_gdf.groupby(zone_col):
            color = color_map.get(zone_name, "#cccccc")
            group.plot(ax=ax2, color=color, edgecolor="black", linewidth=0.5, alpha=0.7)
    else:
        plot_gdf.plot(ax=ax2, edgecolor="black", linewidth=0.5, alpha=0.7)

    ax2.set_xlim(xmin, xmax)
    ax2.set_ylim(ymin, ymax)
    ax2.set_title(f"GeoJSON: {sample_id} (rot={rotation:.1f}\u00b0)", fontsize=14)
    ax2.set_aspect("equal")
    ax2.set_xticks([]); ax2.set_yticks([])
    for spine in ax2.spines.values():
        spine.set_visible(True); spine.set_linewidth(1.5)

    # Stats
    stats_str = (f"Zones: {stats['n_zones']} | Polygons: {stats['n_polygons']} "
                 f"| Lines: {stats['n_lines']} | Features: {stats['n_features']}")
    fig.suptitle(stats_str, fontsize=12, y=0.02)

    # Legend
    legend_patches = []
    for zone_name, style in sorted(zone_styles.items()):
        color = style.get("color", "#cccccc")
        count = style.get("polygon_count", 0)
        legend_patches.append(mpatches.Patch(color=color, label=f"{zone_name} ({count})"))
    if legend_patches:
        ax2.legend(handles=legend_patches, loc="upper left", fontsize=7,
                   bbox_to_anchor=(1.01, 1), title="Zone (poly count)")

    plt.tight_layout()
    plt.show()

    return stats

In [ ]:
# Show a few examples
for sid in ["00000", "00001", "00002"]:
    stats = show_image_and_geojson(sid)
    print(stats)
    print()